# 04. ファクター分析（着順への影響度）
各ファクターが確定着順に与える影響度を定量化し、得点式の再設計に役立てる。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import lightgbm as lgb
import sys; sys.path.append('../src')

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features.csv', encoding='utf-8-sig')
print(f'shape: {df.shape}, races: {df["race_id"].nunique()}')
print('Target 確定着順:', df['確定着順'].describe().round(2))

## 1. 全ファクター Spearman 相関 + 四分位着順差

In [ ]:
FACTORS = {
    '得点系': ['得点', 'デフォルト得点', '得点V1', '得点V2', '得点V3',
              '予想タイム指数', '予想タイム偏差値', '予想タイム指数回帰推定値', 'time_idx_gap',
              '過去5走最高タイム指数', 'タイム指数上昇係数', '勝率', '過程値b'],
    '騎手・調教師': ['騎手評価', '調教師評価', '騎手ランキング', '調教師ランキング',
                   '枠順評価', '脚質評価', 'jockey_trainer_score'],
    '展開・脚質': ['先行指数', '先行率', '予想展開', '波乱度', 'レースレベル'],
    '前走情報': ['前走着順', '前走人気', '前走着差', '前走馬体重', '前走頭数',
               '休養週数', '休養後出走回数', '距離増減', '前走レースレベル'],
    '血統': ['血統距離評価', '血統トラック評価', '血統成長力評価', '血統総合評価'],
    '馬体・状態': ['馬齢', '馬体重', '馬体重増減', 'weight_abs_diff', '負担重量', '斤量比'],
    'レース条件': ['距離', '頭数', 'cond_code', '天候コード', 'is_turf', 'is_dirt'],
    'オッズ系（参考）': ['log_odds', '単勝人気', 'log_odds_gap', 'odds_rank_pct'],
}

records = []
for category, cols in FACTORS.items():
    for col in cols:
        if col not in df.columns:
            continue
        sub = df[['確定着順', col]].dropna()
        if len(sub) < 100:
            continue
        rho, p = stats.spearmanr(sub[col], sub['確定着順'])
        q1_mean = sub[sub[col] <= sub[col].quantile(0.25)]['確定着順'].mean()
        q4_mean = sub[sub[col] >= sub[col].quantile(0.75)]['確定着順'].mean()
        delta = q1_mean - q4_mean
        records.append({'category': category, 'factor': col, 'spearman_rho': round(rho, 3),
                        'p_value': p, 'Q1_avg_pos': round(q1_mean, 2),
                        'Q4_avg_pos': round(q4_mean, 2), 'delta_Q1_Q4': round(delta, 2)})

result_df = pd.DataFrame(records).sort_values('delta_Q1_Q4')
print('=== 着順への影響（四分位着順差: Q1平均 - Q4平均）===')
print('マイナスほど「高値が良い」、プラスほど「低値が良い」')
print(result_df.to_string(index=False))

## 2. 有効ファクター 10分位別平均着順

In [ ]:
key_factors = ['得点', 'デフォルト得点', '予想タイム指数', '予想タイム偏差値',
               '過去5走最高タイム指数', '勝率', '過程値b', '先行指数']
key_factors = [f for f in key_factors if f in df.columns]

ncols = 4
nrows = (len(key_factors) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes_flat = axes.flat if nrows > 1 else [axes] if ncols == 1 else axes

for ax, col in zip(axes_flat, key_factors):
    sub = df[['確定着順', col]].dropna()
    sub = sub.copy()
    sub['decile'] = pd.qcut(sub[col], 10, labels=False, duplicates='drop')
    decile_avg = sub.groupby('decile')['確定着順'].mean()
    decile_avg.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('10分位 (0=低, 9=高)')
    ax.set_ylabel('平均着順')
    ax.axhline(df['確定着順'].mean(), color='red', linestyle='--', linewidth=0.8)

for ax in list(axes_flat)[len(key_factors):]:
    ax.set_visible(False)

plt.suptitle('10分位別平均着順（低いほど良い）', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 3. LightGBM 特徴量重要度（着順予測・オッズなし）

In [ ]:
NON_ODDS_FEATURES = [
    '得点', 'デフォルト得点', '得点V1', '得点V2', '得点V3',
    '予想タイム指数', '予想タイム偏差値', '予想タイム指数回帰推定値', 'time_idx_gap',
    '過去5走最高タイム指数', 'タイム指数上昇係数', '勝率', '過程値b',
    '騎手評価', '調教師評価', '枠順評価', '脚質評価', 'jockey_trainer_score',
    '先行指数', '先行率', '予想展開', 'レースレベル',
    '血統距離評価', '血統トラック評価', '血統総合評価',
    '馬齢', '馬体重', '馬体重増減', 'weight_abs_diff', '負担重量', '斤量比',
    '前走着順', '前走人気', '前走着差', '休養週数', '距離増減', '前走レースレベル',
    '波乱度', '頭数', '距離', 'cond_code', 'is_turf', 'is_dirt',
]
NON_ODDS_FEATURES = [f for f in NON_ODDS_FEATURES if f in df.columns]

df_sorted = df.sort_values('race_id').reset_index(drop=True)
X = df_sorted[NON_ODDS_FEATURES]
y = df_sorted['確定着順']

race_ids = sorted(df_sorted['race_id'].unique())
n_races = len(race_ids)
fold_size = n_races // 5

models = []
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 30,
    'verbose': -1,
    'n_estimators': 500,
    'early_stopping_rounds': 30,
}

for fold in range(5):
    val_races = set(race_ids[fold * fold_size: (fold + 1) * fold_size])
    train_races = set(race_ids[:fold * fold_size])
    if not train_races:
        continue
    tr_mask = df_sorted['race_id'].isin(train_races)
    va_mask = df_sorted['race_id'].isin(val_races)
    model = lgb.LGBMRegressor(**params)
    model.fit(X[tr_mask], y[tr_mask], eval_set=[(X[va_mask], y[va_mask])],
              callbacks=[lgb.log_evaluation(period=200)])
    models.append(model)

fi_gain = pd.Series(
    np.mean([m.booster_.feature_importance('gain') for m in models], axis=0),
    index=NON_ODDS_FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
fi_gain.head(25).plot(kind='barh', ax=ax, title='Feature Importance (Gain) Top25 - 着順予測・オッズなし')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('-- Top 25 Gain --')
print(fi_gain.head(25).to_string())

## 4. 穴馬分析（5番人気以下）
穴馬好走（3着以内）と凡走の各ファクター比較。

In [ ]:
anaba = df[df['単勝人気'] >= 5].copy()
anaba_good = anaba[anaba['確定着順'] <= 3]
anaba_bad  = anaba[anaba['確定着順'] > 3]
fav = df[df['単勝人気'] <= 3].copy()

print(f'穴馬（5番人気以下）総数: {len(anaba)}')
print(f'  好走（3着以内）: {len(anaba_good)} ({len(anaba_good)/len(anaba)*100:.1f}%)')
print(f'  凡走（4着以下）: {len(anaba_bad)}  ({len(anaba_bad)/len(anaba)*100:.1f}%)')

ANABA_FACTORS = [
    '過去5走最高タイム指数', 'タイム指数上昇係数', '予想タイム指数', '予想タイム偏差値',
    '先行指数', '血統総合評価', '騎手評価', '得点', 'デフォルト得点', '勝率', '過程値b',
    '前走着差', '前走着順', '休養週数',
]
ANABA_FACTORS = [f for f in ANABA_FACTORS if f in df.columns]

print('=== 穴馬 好走 vs 凡走 ファクター比較 ===')
print(f'{"Factor":<25} {"好走平均":>10} {"凡走平均":>10} {"差(好-凡)":>10} {"p値":>10}')
print('-' * 70)
for col in ANABA_FACTORS:
    g = anaba_good[col].dropna()
    b = anaba_bad[col].dropna()
    if len(g) < 10 or len(b) < 10:
        continue
    t, p = stats.ttest_ind(g, b)
    diff = g.mean() - b.mean()
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'{col:<25} {g.mean():>10.2f} {b.mean():>10.2f} {diff:>+10.2f} {p:>9.4f}{sig}')

In [ ]:
top_factors = ['過去5走最高タイム指数', '予想タイム指数', '先行指数', '騎手評価']
top_factors = [f for f in top_factors if f in df.columns]

fig, axes = plt.subplots(1, len(top_factors), figsize=(4 * len(top_factors), 5))
if len(top_factors) == 1:
    axes = [axes]

for ax, col in zip(axes, top_factors):
    data = [
        anaba_good[col].dropna().values,
        anaba_bad[col].dropna().values,
        fav[col].dropna().values,
    ]
    ax.boxplot(data, labels=['穴好走', '穴凡走', '人気馬'])
    ax.set_title(col)
    ax.set_ylabel('値')

plt.suptitle('穴好走 vs 穴凡走 vs 人気馬（箱ひげ図）', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 分析まとめ：着順思想の得点式 設計方針

In [ ]:
print('=== 着順への影響 強影響ファクター (|delta| > 3.0) ===')
strong = result_df[result_df['delta_Q1_Q4'].abs() > 3.0][['category','factor','spearman_rho','delta_Q1_Q4']]
print(strong.to_string(index=False))
print()
print('=== 中影響ファクター (1.0 < |delta| <= 3.0) ===')
mid = result_df[result_df['delta_Q1_Q4'].abs().between(1.0, 3.0)][['category','factor','spearman_rho','delta_Q1_Q4']]
print(mid.to_string(index=False))
print()
print('=== 微小影響 (|delta| <= 1.0) ===')
weak = result_df[result_df['delta_Q1_Q4'].abs() <= 1.0][['category','factor','spearman_rho','delta_Q1_Q4']]
print(weak.to_string(index=False))
print()
print('=== 着順思想 得点式 推奨設計 ===')
print('  基盤スコア  : デフォルト得点 + 得点V1 + 得点V2 + 得点V3')
print('  強化推奨    : 予想タイム偏差値（実力の絶対評価、delta大）')
print('  強化推奨    : 過去5走最高タイム指数（穴馬発見に有効）')
print('  強化推奨    : 勝率 / 過程値b（Spearman相関最大クラス）')
print('  追加推奨    : 先行指数（展開利不利 + 穴馬signal）')
print('  削除推奨    : 休養週数ボーナス（着順への影響微小、ROI式向き）')
print('  削除推奨    : 波乱度（レース単位、全馬同一値）')